# Every method, trained and served from Qdrant

s(A→B) is how strongly having skill A implies having skill B. Someone with Spring Boot has Java, and someone with Java may never have used Spring Boot, so a good s is asymmetric. This notebook trains each method from the spike and serves it from Qdrant. Each section shows the method's code as it is in `skillmatch/`, trains it with the config picked on validation, loads it into a Qdrant collection, queries it, and checks that the Qdrant ranking gives the same test MAP as the torch ranking.

The comparison between methods, and what the numbers mean, is in `showcase.ipynb` and `docs/results.md`. This notebook covers the mechanics.

Two directions:

- Up: given a skill, which skills does it imply? Task A asks it for skills never seen in training, Task B for implications hidden from the training graph. Sections 1 to 11.
- Down: given a requirement, who has it? Task D. Section 12.

Qdrant serves a score in one of three ways, and the method decides which:

| Route | When | Methods |
|---|---|---|
| Native distance with HNSW | the score is a dot product or an L1 distance between two stored vectors | cosine, dual encoder, TransE, distilled dual |
| Formula Query over a cosine prefetch | Qdrant has no distance for the geometry, and the score is a short closed form of numbers stored in the payload | cos + gen, order, hyperbolic, any head + gen + cos |
| Shortlist from Qdrant, scored in the client | the score is a neural network, or too long for a formula | box, pair MLP, cross-encoder, LLM judge |

Before running: start Qdrant (`podman run -d -p 6333:6333 docker.io/qdrant/qdrant`) and run from the repo root. The whole notebook takes about 20 minutes on a laptop, mostly for the cross-encoder training and for the Formula Query checks, which rescore the full catalogue on every query.

All numbers here come from one seed (0). The "stored" columns come from `results/` for reference: three-seed means for the trained heads, so they differ by seed noise, one earlier run for the cross-encoder, and the cached answers for the LLM judges, the same answers this notebook reads.

In [2]:
import inspect
import time

import numpy as np
import torch
from IPython.display import Markdown, display
from qdrant_client import QdrantClient, models

from experiments.zoo import FOLDED, config, load, seed_all
from skillmatch import formulas, qdrant
from skillmatch.data import DEVICE, SkillGraph
from skillmatch.methods import (BoxEmbedding, CrossEncoder, DualEncoder, OrderEmbedding, PairMLP, PoincareEmbedding,
                                TransE, cosine, cosine_generality, distill, folded_vectors, hybrid)
from skillmatch.methods.cosine import LAMBDA
from skillmatch.methods.hybrid import tune
from skillmatch.methods.llm_judge import Judged, judge_all, test_tasks
from skillmatch.tasks import down, evaluate, expanded
from skillmatch.training import labelled_pairs, negatives, train_pairwise, train_softmax

graph = SkillGraph()
client = QdrantClient("localhost", port=6333, timeout=300)
ALL = len(graph.catalogue)  # prefetch the whole catalogue, so Qdrant and torch rank the same candidates
COSINE, DOT = models.Distance.COSINE, models.Distance.DOT
DEMO = graph.by_label("Express.js")  # a held-out test skill: it is in no training edge
print(f"{len(graph.ids)} skills, catalogue {ALL}, {len(graph.positives)} training pairs, demo query: {graph.label(DEMO)}")

/Users/andrei/projects/asymmetric-skills/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1106 skills, catalogue 925, 1253 training pairs, demo query: Express.js


In [3]:
STORED = load("recall.json")  # three-seed means of the trained heads
PARITY = {}


def source(*objects):
    """The code of functions or classes, as it is in the repo."""
    display(Markdown("\n".join(f"```python\n{inspect.getsource(o)}```" for o in objects)))


def table(rows):
    """{row: {column: number}} → Markdown table."""
    cols = list(next(iter(rows.values())))
    cell = lambda x: "" if x is None else f"{float(x):.3f}"
    lines = ["| | " + " | ".join(cols) + " |", "|---" * (len(cols) + 1) + "|"]
    display(Markdown("\n".join(lines + [f"| {n} | " + " | ".join(cell(r.get(c)) for c in cols) + " |" for n, r in rows.items()])))


def top(search, a=DEMO, k=5):
    """The k best hits of a Qdrant search for skill a."""
    return ", ".join(f"{graph.label(h.id)} ({h.score:.2f})" for h in search(a)[:k])


def rescored(hits, scores):
    """Qdrant hits with scores computed in the client, best first."""
    return sorted((h.model_copy(update={"score": s}) for h, s in zip(hits, scores)), key=lambda h: -h.score)


def check(name, M, search, stored=None):
    """Test MAP of Tasks A and B: the torch scorer M, and the same method served by Qdrant (search)."""
    with torch.no_grad():
        t, q = (evaluate(graph, S, "test", directions=False) for S in (M, qdrant.served_up(search)))
    stored = stored or (STORED[name]["A MAP"], STORED[name]["B MAP"])
    PARITY[name] = {"A torch": t["A"]["MAP"], "A Qdrant": q["A"]["MAP"], "B torch": t["B"]["MAP"], "B Qdrant": q["B"]["MAP"],
                    "A stored": stored[0], "B stored": stored[1]}
    table({name: PARITY[name]})

## 1. Cosine

s(A→B) = cos(x_A, x_B), where x is the frozen bge-base vector of the skill's text. It gives the same number in both directions, so it is the floor every other method has to beat. Nothing to train.

Qdrant: one dense vector per skill with Cosine distance. The queries here use the encoder rows cached in `data/`, the same vectors the heads read. In production the query text would go through Qdrant inference (`qdrant.document(text)`, section 12).

In [4]:
source(cosine)

```python
def cosine(X):
    return lambda a, b: X[a] @ X[b].T
```

In [5]:
qdrant.catalogue_collection(client, "skills_text", graph, {"text": (graph.X, COSINE)})


def cosine_search(a, limit=ALL):
    return client.query_points("skills_text", query=graph.X[a].tolist(), using="text", limit=limit).points


print("Express.js →", top(cosine_search))
check("cosine", cosine(graph.X), cosine_search)

Express.js → Meteor (0.77), Sinatra (0.76), AngularJS (0.75), Node.js (0.75), React (0.74)


| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| cosine | 0.271 | 0.271 | 0.183 | 0.183 | 0.271 | 0.183 |

## 2. Cosine + generality

s(A→B) = cos(A, B) + λ·gen(B), with gen(B) = log(1 + number of training skills that imply B). Requirements tend to be general skills, and a count of known descendants is a cheap signal of that. λ = 0.05 was picked on validation. Still nothing to train.

Qdrant: `catalogue_collection` stores gen in each point's payload. A Formula Query adds it to the cosine score of a prefetch (`$score`). The optional filter drops skills by id, which the rerankers in sections 10 and 11 need.

In [5]:
source(cosine_generality, formulas.cos_gen)

```python
def cosine_generality(X, generality, lam=LAMBDA):
    return lambda a, b: X[a] @ X[b].T + lam * generality[b]
```
```python
def cos_gen(lam):
    """s(A→B) = cos(A, B) + λ·gen(B): the prefetch score plus the stored generality."""
    return total(["$score", times(lam, "gen")])
```

In [6]:
def cos_gen_search(a, limit=ALL, exclude=()):
    return client.query_points(
        "skills_text",
        prefetch=models.Prefetch(query=graph.X[a].tolist(), using="text", limit=ALL),
        query=models.FormulaQuery(formula=formulas.cos_gen(LAMBDA)),
        query_filter=models.Filter(must_not=[models.HasIdCondition(has_id=list(exclude))]) if exclude else None,
        limit=limit).points


print("Express.js →", top(cos_gen_search))
check("cos+gen", cosine_generality(graph.X, graph.generality), cos_gen_search)

Express.js → JavaScript (0.90), web framework (0.87), JScript (0.86), Java (0.85), JavaScript framework (0.83)


| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| cos+gen | 0.520 | 0.520 | 0.464 | 0.464 | 0.520 | 0.464 |

## 3. Dual encoder

Two small networks on the same text vector, one per role: s(A→B) = src(A)·tgt(B). It is asymmetric because src and tgt differ.

Training (`train_pairwise`, shared by the dual, box, order, TransE and pair MLP heads): each epoch takes every pair (A, B) of the training closure as a positive, plus up to three negatives: the reversed pair (B, A), a random catalogue skill A doesn't imply, and, when there is one, a cousin (another skill below B that A doesn't imply). The head's own loss scores them; for the dual it is binary cross-entropy on the dot product.

In [7]:
source(DualEncoder, negatives, train_pairwise)

```python
class DualEncoder(Head):
    def __init__(self, dim=64):
        super().__init__()
        self.src, self.tgt = text_mlp(dim), text_mlp(dim)

    def forward(self, xa, xb):
        return (self.src(xa) * self.tgt(xb)).sum(-1)

    def matrix(self, xa, xb):
        return self.src(xa) @ self.tgt(xb).T
```
```python
def negatives(graph, a, b):
    out = [(b, a)]
    while True:
        r = random.choice(graph.catalogue)
        if r != a and r not in graph.ancestors_train[a]:
            out.append((a, r))
            break
    cousins = [c for c in graph.descendants_train[b] if c != a and c not in graph.ancestors_train[a]]
    if cousins: out.append((a, random.choice(cousins)))
    return out
```
```python
def train_pairwise(model, graph, epochs, lr, inputs=None, batch=512):
    inputs = graph.X if inputs is None else inputs
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        pairs = labelled_pairs(graph)
        for i in range(0, len(pairs), batch):
            a, b, y = (torch.tensor(t, device=DEVICE) for t in zip(*pairs[i:i + batch]))
            loss = model.loss(model(inputs[a], inputs[b]), y.float())
            opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()
```

In [8]:
kw, epochs, lr = config("dual")  # picked on val: results/sweep_best.json
print(kw, epochs, lr)
seed_all(0)
dual = train_pairwise(DualEncoder(**kw), graph, epochs, lr)

{'dim': 64} 150 0.003


Qdrant: two named vectors per skill, both with Dot distance. The up search sends src(A) against the stored tgt vectors. The same collection answers the reverse question (which skills imply B?) by sending tgt(B) against the stored src vectors.

In [9]:
with torch.no_grad(): src, tgt = dual.src(graph.X), dual.tgt(graph.X)
qdrant.catalogue_collection(client, "skills_dual", graph, {"src": (src, DOT), "tgt": (tgt, DOT)})


def dual_search(a, limit=ALL):  # what does a imply?
    return client.query_points("skills_dual", query=src[a].tolist(), using="tgt", limit=limit).points


def dual_implied_by(b, limit=5):  # which catalogue skills imply b?
    return client.query_points("skills_dual", query=tgt[b].tolist(), using="src", limit=limit).points


print("Express.js →", top(dual_search))
print("→ JavaScript:", ", ".join(graph.label(h.id) for h in dual_implied_by(graph.by_label("JavaScript"))))
check("dual", dual.scorer(graph.X), dual_search)

Express.js → JScript (7.04), web framework (4.95), JVM language (4.12), template processor (3.71), Java (3.04)
→ JavaScript: Puppeteer, Element, Hexo, google-chrome, Silex


| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| dual | 0.375 | 0.375 | 0.377 | 0.377 | 0.371 | 0.355 |

## 4. Box embeddings, + gen + cos

Each skill is an axis-aligned box with soft (Gumbel) edges, and A implies B when A's box sits inside B's: s(A→B) = log P(B | A) = log vol(A ∩ B) − log vol(A), at most 0. Trained with `train_pairwise`; the loss is −log P on positives and −log(1 − P) on negatives.

The heads learn from under a thousand edges and lose some of what the text encoder knows, so every head also comes in a hybrid, head + λ·gen + μ·cos, with (λ, μ) picked on validation (`tune`).

In [10]:
source(BoxEmbedding, hybrid, tune)

```python
class BoxEmbedding(Head):
    def __init__(self, dim=64, beta_i=0.1, beta_v=1.0, n_nodes=None):
        """beta_i, beta_v: Gumbel temperatures of the intersection and the volume. n_nodes: free vectors (ablation)."""
        super().__init__()
        self.beta_i, self.beta_v = beta_i, beta_v
        self.encoder = free_vectors(n_nodes, 2 * dim) if n_nodes else text_mlp(2 * dim)  # centre, half-width

    def embed(self, x):
        centre, half = self.encoder(x).chunk(2, -1)
        half = F.softplus(half)
        return torch.cat([centre - half, centre + half], -1)

    def log_volume(self, lo, hi):
        return torch.log(F.softplus(hi - lo - 2 * EULER_GAMMA * self.beta_v, beta=1 / self.beta_v) + 1e-10).sum(-1)

    def score(self, a, b):
        (lo_a, hi_a), (lo_b, hi_b) = a.chunk(2, -1), b.chunk(2, -1)
        lo = self.beta_i * torch.logaddexp(lo_a / self.beta_i, lo_b / self.beta_i)  # soft max of the lower corners
        hi = -self.beta_i * torch.logaddexp(-hi_a / self.beta_i, -hi_b / self.beta_i)  # soft min of the upper corners
        return (self.log_volume(lo, hi) - self.log_volume(lo_a, hi_a)).clamp(max=-1e-6)

    def loss(self, score, y):
        """score = log P: −log P on positives, −log(1 − P) on negatives."""
        return -(y * score + (1 - y) * torch.log(-torch.expm1(score) + 1e-10)).mean()
```
```python
def hybrid(M, graph, lam, mu):
    return lambda a, b: M(a, b) + lam * graph.generality[b] + mu * (graph.X[a] @ graph.X[b].T)
```
```python
def tune(graph, M, grid=GRID):
    """The (λ, μ) with the best val score."""
    return max(grid, key=lambda w: val_score(graph, hybrid(M, graph, *w)))
```

In [11]:
kw, epochs, lr = config("box")  # results/sweep_best.json
seed_all(0)
box = train_pairwise(BoxEmbedding(**kw), graph, epochs, lr)
with torch.no_grad(): box_w = tune(graph, box.scorer(graph.X))
print(kw, epochs, lr, "(λ, μ) =", box_w)

{'dim': 128, 'beta_i': 0.01} 150 0.001 (λ, μ) = (0.1, 10)


Qdrant has no box distance. The box score can be written as a Formula Query (`formulas.box`, timed below), but at 128 dimensions the formula is about 550 KB of JSON and took about 14 s per query over the full catalogue. So Qdrant returns a cosine shortlist with each skill's box corners in the payload, and the client scores it with the model's own `score`.

In [12]:
with torch.no_grad(): corners = box.embed(graph.X)  # [lo | hi] per skill
qdrant.catalogue_collection(client, "skills_box", graph, {"text": (graph.X, COSINE)}, payload={"box": corners})


def box_search(a, lam=0.0, mu=0.0, limit=ALL):
    """Cosine shortlist from Qdrant, scored here: box score + λ·gen + μ·cos."""
    hits = client.query_points("skills_box", query=graph.X[a].tolist(), using="text", limit=limit,
                               with_payload=["box", "gen"]).points
    stored = torch.tensor([h.payload["box"] for h in hits], device=DEVICE)
    gen, cos = (torch.tensor(v, device=DEVICE) for v in zip(*[(h.payload["gen"], h.score) for h in hits]))
    with torch.no_grad(): s = box.score(corners[a], stored) + lam * gen + mu * cos
    return rescored(hits, s.tolist())


print("Express.js →", top(lambda a: box_search(a, *box_w)))
check("box+gen+cos", hybrid(box.scorer(graph.X), graph, *box_w), lambda a: box_search(a, *box_w))

Express.js → JavaScript framework (7.42), JavaScript (7.32), web framework (7.21), JScript (6.87), Java (5.73)


| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| box+gen+cos | 0.586 | 0.586 | 0.563 | 0.563 | 0.582 | 0.555 |

In [13]:
source(formulas.box, formulas.logaddexp, formulas.softplus)

```python
def box(model, a, key="box"):
    """BoxEmbedding.score: log vol(A ∩ B) − log vol(A), at most −1e-6. payload[key] = B's [lo | hi]."""
    d, bi, bv = len(a) // 2, model.beta_i, model.beta_v
    lo_a, hi_a = a[:d].tolist(), a[d:].tolist()
    log_vol_a = model.log_volume(a[:d], a[d:]).item()
    log_sides = []
    for i in range(d):
        lo = times(bi, logaddexp(lo_a[i] / bi, times(1 / bi, field(key, i))))  # soft max of the lower corners
        hi = times(-bi, logaddexp(-hi_a[i] / bi, times(-1 / bi, field(key, d + i))))  # soft min of the upper corners
        side = total([hi, times(-1, lo), -2 * EULER_GAMMA * bv])
        log_sides.append(ln(total([times(bv, softplus(times(1 / bv, side))), 1e-10])))
    return minimum(total([*log_sides, -log_vol_a]), -1e-6)
```
```python
def logaddexp(c, x):
    """log(e^c + e^x) for a constant c: max(c, x) + ln(1 + e^−|x − c|)."""
    return total([maximum(x, c), log1p_exp_neg_abs(total([x, -c]))])
```
```python
def softplus(x):
    """log(1 + e^x) = max(0, x) + ln(1 + e^−|x|)."""
    return total([relu(x), log1p_exp_neg_abs(x)])
```

In [14]:
# The same box + gen + cos as a Formula Query on a 100-skill prefetch, against the client-side route on the same 100
formula = formulas.hybrid(formulas.box(box, corners[DEMO]), *box_w)
t = time.perf_counter()
server = client.query_points("skills_box", prefetch=models.Prefetch(query=graph.X[DEMO].tolist(), using="text", limit=100),
                             query=models.FormulaQuery(formula=formula), limit=10).points
t_server = time.perf_counter() - t
t = time.perf_counter()
local = box_search(DEMO, *box_w, limit=100)[:10]
t_local = time.perf_counter() - t
print(f"Formula Query: {t_server:.2f} s, client-side: {t_local:.3f} s, same top 10: {[h.id for h in server] == [h.id for h in local]}, "
      f"max score gap {max(abs(s.score - l.score) for s, l in zip(server, local)):.1e}")

Formula Query: 1.54 s, client-side: 0.041 s, same top 10: True, max score gap 2.2e-06


## 5. Order embeddings, + gen + cos

Each skill is a point with non-negative coordinates, specific skills further from the origin, and A implies B when A ≥ B on every coordinate. s(A→B) = −Σᵢ max(0, f(B)ᵢ − f(A)ᵢ)², the order violation, which is 0 when A sits above B everywhere. Trained with `train_pairwise` and a max-margin loss. Order + gen + cos has the best Task A MAP of the trained heads, 0.618 over three seeds.

Qdrant: f(B) goes in the payload as an array, and a Formula Query computes the violation over a cosine prefetch. The max becomes (x + |x|) / 2, since formulas have abs but no max. The head, gen and cos are then one server-side query.

In [15]:
source(OrderEmbedding, formulas.order, formulas.relu, formulas.hybrid)

```python
class OrderEmbedding(Head):
    def __init__(self, dim=64, margin=1.0, n_nodes=None):
        super().__init__()
        self.margin = margin
        self.encoder = free_vectors(n_nodes, dim) if n_nodes else text_mlp(dim)

    def embed(self, x):
        return self.encoder(x).abs()

    def dist(self, a, b):
        return F.relu(b - a).pow(2).sum(-1)

    def score(self, a, b):
        return -self.dist(a, b)

    def loss(self, score, y):
        return max_margin_loss(score, y, self.margin)
```
```python
def order(a, key="order"):
    """OrderEmbedding.score: −Σᵢ max(0, bᵢ − aᵢ)²."""
    return times(-1, total(square(relu(total([field(key, i), -v]))) for i, v in enumerate(a.tolist())))
```
```python
def relu(x):
    """max(0, x) = (x + |x|) / 2."""
    return times(0.5, total([x, absolute(x)]))
```
```python
def hybrid(head, lam, mu):
    """s(A→B) = head + λ·gen(B) + μ·cos(A, B), as methods.hybrid."""
    return total([head, times(lam, "gen"), times(mu, "$score")])
```

In [16]:
kw, epochs, lr = config("order")  # results/compare_sweep.json
seed_all(0)
order = train_pairwise(OrderEmbedding(**kw), graph, epochs, lr)
with torch.no_grad(): order_w = tune(graph, order.scorer(graph.X))
print(kw, epochs, lr, "(λ, μ) =", order_w)

{'dim': 128, 'margin': 0.1} 150 0.001 (λ, μ) = (0.1, 3)


In [17]:
with torch.no_grad(): f_order = order.embed(graph.X)
qdrant.catalogue_collection(client, "skills_order", graph, {"text": (graph.X, COSINE)}, payload={"order": f_order})


def order_search(a, lam=0.0, mu=0.0, limit=ALL, prefetch=ALL):
    return client.query_points(
        "skills_order",
        prefetch=models.Prefetch(query=graph.X[a].tolist(), using="text", limit=prefetch),
        query=models.FormulaQuery(formula=formulas.hybrid(formulas.order(f_order[a]), lam, mu)),
        limit=limit).points


print("Express.js →", top(lambda a: order_search(a, *order_w)))
check("order+gen+cos", hybrid(order.scorer(graph.X), graph, *order_w), lambda a: order_search(a, *order_w))

Express.js → JavaScript (2.49), web framework (2.43), JavaScript framework (2.38), JScript (2.36), Java (2.30)


| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| order+gen+cos | 0.626 | 0.626 | 0.548 | 0.548 | 0.618 | 0.529 |

## 6. Hyperbolic embeddings, with the is-a score

Poincaré embeddings (Nickel and Kiela 2017), the geometry of the Qdrant hyperbolic article. Skills are points in the unit ball and general skills settle near the origin. The distance d is symmetric, so the article's score −d has no direction. The is-a score adds one: s(A→B) = −(1 + α(‖B‖ − ‖A‖))·d(A, B) also asks B to sit closer to the origin than A. α = 0.1 was picked on validation.

Training (`train_softmax`, the Nickel and Kiela loss): a softmax over −d that has to pick B out of B plus ten skills unrelated to A.

Qdrant: the coordinates go in the payload with ‖B‖² and ‖B‖, and the formula writes acosh as ln(x + √(x² − 1)), as in the article.

In [18]:
source(PoincareEmbedding, train_softmax, formulas.poincare)

```python
class PoincareEmbedding(Head):
    def __init__(self, dim=5, alpha=0.0, n_nodes=None):
        super().__init__()
        self.alpha = alpha
        self.encoder = free_vectors(n_nodes, dim, init=1e-3) if n_nodes else text_mlp(dim)  # N&K init near the origin

    def embed(self, x):
        return expmap0(self.encoder(x))

    def dist(self, a, b):
        return poincare_distance(a, b)

    def score(self, a, b):
        return -(1 + self.alpha * (b.norm(dim=-1) - a.norm(dim=-1))) * self.dist(a, b)
```
```python
def train_softmax(model, graph, epochs, lr, inputs=None, k=10, batch=64):
    """Needs model.embed and model.dist. ponytail: Adam on tangent vectors (expmap0) instead of N&K's Riemannian
    SGD with burn-in; switch if the fit stalls."""
    inputs = graph.X if inputs is None else inputs
    related = {a: graph.ancestors_train[a] | set(graph.descendants_train[a]) | {a} for a in graph.catalogue}

    def unrelated(a):
        out = []
        while len(out) < k:
            r = random.choice(graph.catalogue)
            if r not in related[a]: out.append(r)
        return out

    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        shuffled = random.sample(graph.positives, len(graph.positives))
        for i in range(0, len(shuffled), batch):
            a, b = zip(*shuffled[i:i + batch])
            candidates = torch.tensor([[pb] + unrelated(pa) for pa, pb in zip(a, b)], device=DEVICE)  # true B first
            d = model.dist(model.embed(inputs[torch.tensor(a, device=DEVICE)])[:, None], model.embed(inputs[candidates]))
            loss = F.cross_entropy(-d, torch.zeros(len(a), dtype=torch.long, device=DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()
```
```python
def poincare(model, a, key="hyp"):
    """PoincareEmbedding.score: −(1 + α(‖B‖ − ‖A‖))·d(A, B), d = acosh(1 + 2‖A − B‖² / ((1 − ‖A‖²)(1 − ‖B‖²))).
    payload: key = B's coordinates, key + "_sq" = min(‖B‖², 1 − 1e-5), key + "_norm" = ‖B‖."""
    uu = min((a @ a).item(), 1 - 1e-5)
    dist2 = total(square(total([field(key, i), -v])) for i, v in enumerate(a.tolist()))
    denominator = times(1 - uu, total([1, times(-1, f"{key}_sq")]))
    x = maximum(total([1, models.DivExpression(div=models.DivParams(left=times(2, dist2), right=denominator))]), 1 + 1e-7)
    d = ln(total([x, models.SqrtExpression(sqrt=total([square(x), -1]))]))  # acosh
    return times(-1, total([1, times(model.alpha, total([f"{key}_norm", -a.norm().item()]))]), d)
```

In [19]:
kw, epochs, lr = config("hyp+isa")  # results/compare_sweep.json; alpha only changes the score, not the training
seed_all(0)
hyp = train_softmax(PoincareEmbedding(**kw), graph, epochs, lr)
print(kw, epochs, lr)

{'dim': 10, 'alpha': 0.1} 150 0.003


In [20]:
with torch.no_grad(): p_hyp = hyp.embed(graph.X)
qdrant.catalogue_collection(client, "skills_hyp", graph, {"text": (graph.X, COSINE)},
                            payload={"hyp": p_hyp, "hyp_sq": (p_hyp ** 2).sum(-1).clamp(max=1 - 1e-5), "hyp_norm": p_hyp.norm(dim=-1)})


def hyp_search(a, limit=ALL):
    return client.query_points(
        "skills_hyp",
        prefetch=models.Prefetch(query=graph.X[a].tolist(), using="text", limit=ALL),
        query=models.FormulaQuery(formula=formulas.poincare(hyp, p_hyp[a])),
        limit=limit).points


print("Express.js →", top(hyp_search))
check("hyp+isa", hyp.scorer(graph.X), hyp_search)

Express.js → JScript (-13.22), web framework (-13.43), JavaScript (-14.07), JVM language (-14.41), Java (-14.53)


| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| hyp+isa | 0.401 | 0.401 | 0.437 | 0.437 | 0.422 | 0.460 |

## 7. TransE, + gen + cos

One relation r, "implies": s(A→B) = −‖f(A) + r − f(B)‖₁ (the L1 norm was picked on validation). Trained with `train_pairwise` and a max-margin loss.

Qdrant: TransE alone is a native search. Store f(B) with Manhattan distance and send f(A) + r. Qdrant returns the distance, smaller first, so the search negates it to get s. The hybrid also needs gen and cos, so it keeps f(B) in the payload as well and computes the distance in a formula over a cosine prefetch.

In [21]:
source(TransE, formulas.transe)

```python
class TransE(Head):
    def __init__(self, dim=64, margin=1.0, p=2, n_nodes=None):
        super().__init__()
        self.margin, self.p = margin, p
        self.encoder = free_vectors(n_nodes, dim) if n_nodes else text_mlp(dim)
        self.r = nn.Parameter(0.1 * torch.randn(dim))

    def embed(self, x):
        return self.encoder(x)

    def dist(self, a, b):
        return (a + self.r - b).norm(p=self.p, dim=-1)

    def score(self, a, b):
        return -self.dist(a, b)

    def loss(self, score, y):
        return max_margin_loss(score, y, self.margin)
```
```python
def transe(a, model, key="transe"):
    """TransE.score: −‖f(A) + r − f(B)‖_p, a = f(A). payload[key] = f(B)."""
    diffs = [total([field(key, i), -v]) for i, v in enumerate((a + model.r).tolist())]
    return times(-1, total(map(absolute, diffs)) if model.p == 1 else models.SqrtExpression(sqrt=total(map(square, diffs))))
```

In [22]:
kw, epochs, lr = config("transe")  # results/compare_sweep.json
seed_all(0)
transe = train_pairwise(TransE(**kw), graph, epochs, lr)
with torch.no_grad(): transe_w = tune(graph, transe.scorer(graph.X))
print(kw, epochs, lr, "(λ, μ) =", transe_w)

{'dim': 128, 'margin': 1.0, 'p': 1} 150 0.001 (λ, μ) = (0.1, 3)


In [23]:
with torch.no_grad(): f_transe = transe.embed(graph.X)
distance = models.Distance.MANHATTAN if transe.p == 1 else models.Distance.EUCLID
qdrant.catalogue_collection(client, "skills_transe", graph, {"text": (graph.X, COSINE), "transe": (f_transe, distance)},
                            payload={"transe": f_transe})


def transe_search(a, limit=ALL):
    with torch.no_grad(): query = f_transe[a] + transe.r
    hits = client.query_points("skills_transe", query=query.tolist(), using="transe", limit=limit).points
    return [h.model_copy(update={"score": -h.score}) for h in hits]  # Qdrant returns the distance


def transe_gen_cos_search(a, lam, mu, limit=ALL):
    return client.query_points(
        "skills_transe",
        prefetch=models.Prefetch(query=graph.X[a].tolist(), using="text", limit=ALL),
        query=models.FormulaQuery(formula=formulas.hybrid(formulas.transe(f_transe[a], transe), lam, mu)),
        limit=limit).points


print("Express.js →", top(transe_search))
check("transe", transe.scorer(graph.X), transe_search)
check("transe+gen+cos", hybrid(transe.scorer(graph.X), graph, *transe_w), lambda a: transe_gen_cos_search(a, *transe_w))

Express.js → JavaScript (-0.16), JScript (-0.17), JavaScript framework (-0.20), web framework (-0.21), PHP (-0.23)


| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| transe | 0.209 | 0.209 | 0.352 | 0.352 | 0.214 | 0.356 |

| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| transe+gen+cos | 0.544 | 0.544 | 0.525 | 0.525 | 0.576 | 0.524 |

## 8. Pair MLP

A classifier on the ordered pair: MLP([x_A, x_B, x_A ⊙ x_B, x_A − x_B]) → the logit of P(B | A). It has no vector per skill, so Qdrant can only supply candidates: a cosine shortlist with the stored text vectors, scored in the client. With the whole catalogue as the shortlist, the ranking equals the torch one. A deployment would score a shorter list: the cos + gen top 50 holds 99% of the Task A answers and 93% of the Task B answers.

In [24]:
source(PairMLP)

```python
class PairMLP(Head):
    def __init__(self, hidden=512, in_dim=TEXT_DIM):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4 * in_dim, hidden), nn.GELU(), nn.Linear(hidden, 1))

    def forward(self, xa, xb):
        return self.net(torch.cat([xa, xb, xa * xb, xa - xb], -1)).squeeze(-1)

    def matrix(self, xa, xb, chunk=16):
        return torch.cat([self(xa[i:i + chunk, None].expand(-1, len(xb), -1), xb[None].expand(len(xa[i:i + chunk]), -1, -1))
                          for i in range(0, len(xa), chunk)])
```

In [25]:
kw, epochs, lr = config("pair_mlp")  # results/compare_sweep.json
seed_all(0)
mlp = train_pairwise(PairMLP(**kw), graph, epochs, lr)
print(kw, epochs, lr)

{'hidden': 512} 150 0.001


In [26]:
def pair_mlp_search(a, limit=ALL):
    hits = client.query_points("skills_text", query=graph.X[a].tolist(), using="text", limit=limit, with_vectors=["text"]).points
    xb = torch.tensor([h.vector["text"] for h in hits], device=DEVICE)
    with torch.no_grad(): s = mlp(graph.X[a].expand(len(xb), -1), xb)
    return rescored(hits, s.tolist())


print("Express.js →", top(pair_mlp_search))
check("pair_mlp", mlp.scorer(graph.X), pair_mlp_search)

Express.js → JScript (4.20), JVM language (3.20), web framework (3.07), JavaScript framework (2.98), Java (2.90)


| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| pair_mlp | 0.520 | 0.520 | 0.417 | 0.417 | 0.521 | 0.411 |

## 9. Distilled dual, with gen + cos folded in

Box + gen + cos ranks well, but it needs a shortlist and client-side scoring. Distillation trains a dual encoder to reproduce its ranking: for each catalogue skill, the student's softmax over the catalogue matches the teacher's (listwise KL). Only catalogue skills are queries and candidates, so held-out skills and hidden edges never reach the student.

With the fold, the student learns only the box part and keeps gen and cos from the teacher's (λ, μ). The score stays one dot product, [src(A), √μ·x_A, 1] · [tgt(B), √μ·x_B, λ·gen(B)], so Qdrant serves it natively with Dot distance.

In [27]:
source(distill, folded_vectors)

```python
def distill(graph, teacher, dim=64, tau=1.0, epochs=150, lr=1e-3, batch=64, fold=None):
    """Returns (student, its scorer)."""
    catalogue = torch.tensor(graph.catalogue)
    with torch.no_grad():
        T = torch.cat([teacher(catalogue[i:i + 256], catalogue) for i in range(0, len(catalogue), 256)])
        P = F.softmax(T.fill_diagonal_(-float("inf")) / tau, -1)
    student = DualEncoder(dim).to(DEVICE)
    opt = torch.optim.Adam(student.parameters(), lr=lr)
    S = student.scorer(graph.X) if fold is None else hybrid(student.scorer(graph.X), graph, *fold)
    for _ in range(epochs):
        for q in torch.randperm(len(catalogue)).split(batch):
            loss = F.kl_div(F.log_softmax(S(catalogue[q], catalogue), -1), P[q.to(DEVICE)], reduction="batchmean")
            opt.zero_grad(); loss.backward(); opt.step()
    return student.eval(), S
```
```python
def folded_vectors(student, x, generality, lam, mu):
    """(query, document) vectors whose dot product is the folded score: query for the A side, document for B."""
    ones = torch.ones(len(x), 1, device=x.device)
    query = torch.cat([student.src(x), mu ** 0.5 * x, ones], -1)
    document = torch.cat([student.tgt(x), mu ** 0.5 * x, lam * generality[:, None]], -1)
    return query, document
```

In [28]:
seed_all(0)
teacher = hybrid(box.scorer(graph.X), graph, *box_w)  # box + gen + cos from section 4
student, folded = distill(graph, teacher, **FOLDED, fold=box_w)
print(FOLDED, "fold (λ, μ) =", box_w)

{'dim': 64, 'tau': 1.0, 'epochs': 300} fold (λ, μ) = (0.1, 10)


In [29]:
with torch.no_grad(): query, document = folded_vectors(student, graph.X, graph.generality, *box_w)
qdrant.catalogue_collection(client, "skills_folded", graph, {"folded": (document, DOT)})


def folded_search(a, limit=ALL):
    return client.query_points("skills_folded", query=query[a].tolist(), using="folded", limit=limit).points


print("Express.js →", top(folded_search))
check("distilled dual, folded", folded, folded_search)

Express.js → JavaScript (-9.46), JavaScript framework (-9.66), JScript (-10.28), web framework (-10.34), Node.js (-10.84)


| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| distilled dual, folded | 0.493 | 0.493 | 0.577 | 0.577 | 0.496 | 0.531 |

## 10. Cross-encoder reranker

`bge-reranker-base` fine-tuned on the same labelled pairs as the heads. It reads the two texts together, in order, so it has direction, but it costs one transformer pass per pair and can only rerank a shortlist.

Qdrant supplies the shortlist: the cos + gen top 50, minus the skills the training graph already says A implies (a `HasIdCondition` filter). It is the same shortlist the LLM judges rerank in section 11. The 50 go first, ordered by cos + gen + p(A→B), the blend `Judged` uses; the rest keep their cos + gen order.

In [30]:
def known(a):
    """a itself and the skills the training graph says a implies: Task B filters them out."""
    return graph.ancestors_train.get(a, set()) | {a}


def shortlist(a, k=50):
    return [h.id for h in cos_gen_search(a, limit=k, exclude=known(a))]


def judged_search(p):
    """p: {A: {B: p(A→B)}} for the shortlist of each A. Qdrant's cos + gen list with the judged skills on top."""
    def search(a):
        hits, pa = cos_gen_search(a, exclude=known(a)), p.get(a, {})
        return rescored(hits, [h.score + 10 + pa[h.id] if h.id in pa else h.score for h in hits])
    return search


tasks, _ = test_tasks(graph)  # the torch shortlists behind results/llm_judge.json
same = sum(set(shortlist(a)) == set(c) for a, c in tasks)
print(f"Qdrant shortlist = torch shortlist for {same} of {len(tasks)} test queries")
assert same == len(tasks)

Qdrant shortlist = torch shortlist for 213 of 213 test queries


In [31]:
source(CrossEncoder)

```python
class CrossEncoder:
    def __init__(self, texts, name="BAAI/bge-reranker-base", max_len=96):
        """texts: the text of every node, by index."""
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.texts, self.max_len = texts, max_len
        self.tokenizer = AutoTokenizer.from_pretrained(name)
        self.model = AutoModelForSequenceClassification.from_pretrained(name).to(DEVICE)
        self.opt = None

    def score_texts(self, texts_a, texts_b):
        enc = self.tokenizer(texts_a, texts_b, truncation=True, max_length=self.max_len, padding=True, return_tensors="pt")
        return self.model(**enc.to(DEVICE)).logits.squeeze(-1)

    def logits(self, a, b):
        return self.score_texts([self.texts[i] for i in a], [self.texts[j] for j in b])

    def pairs(self, a, b, batch=256):
        """Aligned pairs, inference. fp16 on MPS: 2.5× faster, logits within 0.06 of fp32."""
        self.model.eval()
        with torch.no_grad(), torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "mps"):
            return torch.cat([self.logits(a[i:i + batch].tolist(), b[i:i + batch].tolist()).float() for i in range(0, len(a), batch)])

    def __call__(self, a, b):
        return self.pairs(a.repeat_interleave(len(b)), b.repeat(len(a))).view(len(a), len(b))

    def fit_epoch(self, labelled, lr=2e-5, batch=32):
        """labelled: [(A id, B id, y)], e.g. training.labelled_pairs(graph)."""
        if self.opt is None: self.opt = torch.optim.AdamW(self.model.parameters(), lr=lr)
        self.model.train()
        for i in range(0, len(labelled), batch):
            a, b, y = zip(*labelled[i:i + batch])
            loss = F.binary_cross_entropy_with_logits(self.logits(a, b), torch.tensor(y, device=DEVICE))
            self.opt.zero_grad(); loss.backward(); self.opt.step()
```

In [32]:
cfg, epochs, lr = load("compare.json")["cross_enc"]["config"]  # epochs picked on val (experiments/compare.py ce)
print(cfg, epochs, lr)
seed_all(0)
ce = CrossEncoder([graph.text(i) for i in range(len(graph.ids))], cfg["model"], cfg["max_len"])
t = time.time()
for _ in range(epochs): ce.fit_epoch(labelled_pairs(graph), lr)
print(f"trained in {time.time() - t:.0f} s")

{'model': 'BAAI/bge-reranker-base', 'lr': 2e-05, 'max_len': 96} 3 2e-05


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trained in 205 s


In [33]:
def p_cross_encoder(a, ids):
    return torch.sigmoid(ce.pairs(torch.tensor([a] * len(ids)), torch.tensor(ids))).tolist()


ce_p = {a: dict(zip(ids, p_cross_encoder(a, ids))) for a, _ in tasks for ids in [shortlist(a)]}
stored = load("compare.json")["cross_enc rerank50"]
check("cross-encoder, top 50", Judged(graph, ce_p, {}), judged_search(ce_p), (stored["A"]["MAP"], stored["B"]["MAP"]))

| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| cross-encoder, top 50 | 0.566 | 0.566 | 0.485 | 0.483 | 0.601 | 0.480 |

## 11. LLM judges

Two LLMs from different model families score p(A→B) for the same shortlists: Claude Sonnet 5 and Codex. The prompt is below. Claude wrote the graph's labels, so a Claude judge alone would be circular. The answers are cached in `data/llm_cache/` by prompt hash, so this cell makes no LLM calls. The reranking is the same `Judged` blend as for the cross-encoder.

In [34]:
from skillmatch.methods.llm_judge import rank_prompt
print(rank_prompt(graph, tasks[:1])[:1500])

p(A→B) = probability that a person with solid working experience in skill A has at least basic working competence in skill B, because working with A requires or exercises B. Judge real-world practice, not mere co-occurrence.

For each query skill A, score p(A→B) for every candidate B.
Answer with one JSON object only: {"Q1": {"1": p, "2": p, ...}, "Q2": {...}}, p in [0, 1], two decimals.

Q1. A = Excelize — Golang library for reading and writing spreadsheet files.
  1. Python — general-purpose programming language
  2. JVM language — (category)
  3. JavaScript — high-level programming language
  4. Java — object-oriented programming language
  5. JScript — implementation of ECMAScript by Microsoft
  6. web framework — (category)
  7. NumPy — numerical programming package for the Python programming language
  8. ruby — general-purpose programming language
  9. C++ — general-purpose programming language
  10. Hypertext Transfer Protocol — application-layer protocol for transferring web r

In [35]:
llm = load("llm_judge.json")
for judge in ("claude-sonnet-5", "codex-gpt-6-astra"):
    rank, pair, missing = judge_all(graph, judge)
    check(f"LLM {judge}, top 50", Judged(graph, rank, pair), judged_search(rank), (llm[judge]["A"]["MAP"], llm[judge]["B"]["MAP"]))

| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| LLM claude-sonnet-5, top 50 | 0.773 | 0.773 | 0.698 | 0.698 | 0.773 | 0.698 |

| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| LLM codex-gpt-6-astra, top 50 | 0.757 | 0.757 | 0.686 | 0.686 | 0.757 | 0.686 |

## 12. Down: who has a requirement

Every section above answers the up question. A recruiter asks the down question: given requirement B, rank the profiles by s(profile→B). Scoring every profile directly by s(profile→B) gave test MAP between 0.02 and 0.37 (Task D in `docs/results.md`). Ingest-time expansion scored higher for every model: when a profile arrives, the scorer lists its top K implied catalogue skills once, and Qdrant stores the list as a sparse vector (index = skill id, value = 926 − rank). A requirement is then a sparse query {b: 1}, and several requirements add up. Ingesting one CV rescores nobody.

This route also uses Qdrant inference: `qdrant.document(text)` hands the text to Qdrant, which embeds it with FastEmbed (inside the client here, on the server in Qdrant Cloud). The profiles are the 925 catalogue skills plus the 110 held-out test skills.

In [36]:
source(qdrant.expansions, qdrant.ingest_profiles, qdrant.link, qdrant.who_has)

```python
def expansions(graph, M, profiles, ks):
    """Profiles → {K: [sparse vector of each profile's top K implied catalogue skills]}."""
    catalogue = torch.tensor(graph.catalogue)
    order = M(torch.tensor(profiles), catalogue).argsort(-1, descending=True).cpu()
    value = (len(catalogue) - torch.arange(len(catalogue))).tolist()  # value of the skill at each rank
    return {k: [models.SparseVector(indices=catalogue[o[:k]].tolist(), values=value[:k]) for o in order] for k in ks}
```
```python
def ingest_profiles(client, graph, scorers, ks, profiles, texts, payloads, batch=256):
    """Upsert profiles (ids = node ids, or new ids past them for CVs added with graph.add_texts)."""
    vectors = [{"text": document(t)} for t in texts]
    for name, M in scorers.items():
        for k, sparse in expansions(graph, M, profiles, ks).items():
            for v, sv in zip(vectors, sparse): v[f"{name}@{k}"] = sv
    for i in range(0, len(profiles), batch):
        client.upsert("profiles", [models.PointStruct(id=p, vector=v, payload=pl)
                                   for p, v, pl in zip(profiles[i:i + batch], vectors[i:i + batch], payloads[i:i + batch])])
```
```python
def link(client, requirement, k=1):
    """Requirement text → catalogue skill ids, by dense search (Qdrant inference embeds the query)."""
    return [p.id for p in client.query_points("skills", query=document(requirement), using="text", limit=k).points]
```
```python
def who_has(client, skills, using, limit=10, **kw):
    """Profiles whose stored expansion contains the skills, best first. kw → query_points (e.g. query_filter)."""
    query = models.SparseVector(indices=list(skills), values=[1.0] * len(skills))
    return client.query_points("profiles", query=query, using=using, limit=limit, with_payload=True, **kw).points
```

In [37]:
profiles = graph.catalogue + graph.heldout["test"]
order_gen_cos = hybrid(order.scorer(graph.X), graph, *order_w)  # section 5
t = time.time()
with torch.no_grad(): qdrant.build(client, graph, {"order+gen+cos": order_gen_cos}, [100], profiles)
print(f"ingested {len(profiles)} profiles in {time.time() - t:.0f} s")

skills = qdrant.link(client, "JavaScript")  # requirement text → catalogue skill
print("linked to:", graph.label(skills[0]))
for h in qdrant.who_has(client, skills, "order+gen+cos@100", limit=6):  # 925 = the profile's top implied skill; Qdrant returns ties in id order
    print(f"  {h.payload['label']} ({h.payload['kind']}, {h.score:.0f})")

ingested 1035 profiles in 76 s
linked to: JavaScript
  JavaScript (catalogue, 925)
  jsPDF (unseen, 925)
  Liquid (unseen, 925)
  Express.js (unseen, 925)
  Three.js (unseen, 925)
  SigmaJS (unseen, 925)


In [38]:
served = qdrant.served(lambda b: qdrant.who_has(client, [b], "order+gen+cos@100", len(profiles)))
with torch.no_grad(): rows = {"torch, full ranking": down(graph, expanded(graph, order_gen_cos)), "Qdrant, K = 100": down(graph, served)}
stored = load("qdrant.json")
rows["stored run, Qdrant, K = 100"] = stored["order+gen+cos@100"]
table({n: {k: r[k] for k in ("MAP", "R@10", "R@50", "R@100")} for n, r in rows.items()})

| | MAP | R@10 | R@50 | R@100 |
|---|---|---|---|---|
| torch, full ranking | 0.491 | 0.622 | 0.816 | 0.904 |
| Qdrant, K = 100 | 0.491 | 0.622 | 0.801 | 0.889 |
| stored run, Qdrant, K = 100 | 0.491 | 0.622 | 0.801 | 0.889 |

## 13. Prefetch size

The checks above prefetch the whole catalogue, so Qdrant ranks the same candidates as torch. A deployment prefetches fewer. This cell measures what a shorter cosine prefetch costs order + gen + cos, and the time per query (one Qdrant container on a laptop, queries sent one at a time).

In [39]:
rows = {}
for L in (50, 100, 200, ALL):
    ms = []

    def timed(a, L=L):
        t = time.perf_counter()
        hits = order_search(a, *order_w, limit=L, prefetch=L)
        ms.append(1000 * (time.perf_counter() - t))
        return hits

    with torch.no_grad(): r = evaluate(graph, qdrant.served_up(timed), "test", directions=False)
    rows[f"prefetch {L}"] = {"A MAP": r["A"]["MAP"], "B MAP": r["B"]["MAP"], "p50 ms": np.median(ms)}
table(rows)

| | A MAP | B MAP | p50 ms |
|---|---|---|---|
| prefetch 50 | 0.526 | 0.435 | 42.229 |
| prefetch 100 | 0.586 | 0.483 | 64.580 |
| prefetch 200 | 0.600 | 0.511 | 104.171 |
| prefetch 925 | 0.626 | 0.548 | 383.581 |

## 14. Qdrant against torch

One row per served method. "torch" is the method's own scorer over the full catalogue, "Qdrant" the same method through its collection. The two should agree up to float rounding, since the formulas run in Qdrant and the vectors pass through JSON. The stored columns are the reference runs from `results/` described at the top.

In [40]:
table(PARITY)
gap = max(abs(r[f"{t} torch"] - r[f"{t} Qdrant"]) for r in PARITY.values() for t in "AB")
print(f"largest torch/Qdrant MAP gap: {gap:.4f}")
assert gap < 0.005

| | A torch | A Qdrant | B torch | B Qdrant | A stored | B stored |
|---|---|---|---|---|---|---|
| cosine | 0.271 | 0.271 | 0.183 | 0.183 | 0.271 | 0.183 |
| cos+gen | 0.520 | 0.520 | 0.464 | 0.464 | 0.520 | 0.464 |
| dual | 0.375 | 0.375 | 0.377 | 0.377 | 0.371 | 0.355 |
| box+gen+cos | 0.586 | 0.586 | 0.563 | 0.563 | 0.582 | 0.555 |
| order+gen+cos | 0.626 | 0.626 | 0.548 | 0.548 | 0.618 | 0.529 |
| hyp+isa | 0.401 | 0.401 | 0.437 | 0.437 | 0.422 | 0.460 |
| transe | 0.209 | 0.209 | 0.352 | 0.352 | 0.214 | 0.356 |
| transe+gen+cos | 0.544 | 0.544 | 0.525 | 0.525 | 0.576 | 0.524 |
| pair_mlp | 0.520 | 0.520 | 0.417 | 0.417 | 0.521 | 0.411 |
| distilled dual, folded | 0.493 | 0.493 | 0.577 | 0.577 | 0.496 | 0.531 |
| cross-encoder, top 50 | 0.566 | 0.566 | 0.485 | 0.483 | 0.601 | 0.480 |
| LLM claude-sonnet-5, top 50 | 0.773 | 0.773 | 0.698 | 0.698 | 0.773 | 0.698 |
| LLM codex-gpt-6-astra, top 50 | 0.757 | 0.757 | 0.686 | 0.686 | 0.757 | 0.686 |

largest torch/Qdrant MAP gap: 0.0015


## What this notebook doesn't test

- Scale. Every collection holds 925 skills and the profile collection 1035 profiles. HNSW recall and formula latency at a million points are not measured.
- Concurrency. Latency comes from queries sent one at a time to a local container.
- Qdrant Cloud inference. Section 12 embeds through FastEmbed inside the client; the up sections send the cached encoder rows.
- The box Formula Query beyond one query. It matched the client-side scores on the demo query, and its latency rules it out at this dimension.
- Why the cross-encoder scores lower here than in `results/compare.json`. Both runs use the same training code, seed 0 and three epochs. The stored run also evaluated between epochs, and training on MPS isn't bit-reproducible. The cause of the gap is untested.
- Free-vector ablations, neighbour votes and the Euclidean twin of the hyperbolic model are left out: they are ablations, not methods to serve. They are in `experiments/`.